<a href="https://colab.research.google.com/github/ManasaRPatel/MachineLearning2_LAB/blob/main/BayesianNetworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pgmpy numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]


In [9]:
import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

# 1. Create a dummy dataset (representing: Rain -> WetGrass, Sprinkler -> WetGrass)
# We'll generate random data for Rain (0=No, 1=Yes), Sprinkler (0=No, 1=Yes), and WetGrass (0=No, 1=Yes)
np.random.seed(42)
n_samples = 1000

rain = np.random.binomial(1, 0.3, n_samples)
sprinkler = np.random.binomial(1, 0.4, n_samples)
# WetGrass is more likely if it rains or if the sprinkler is on
wet_grass_prob = 0.1 + 0.8 * rain + 0.7 * sprinkler - 0.6 * rain * sprinkler
wet_grass = np.random.binomial(1, np.clip(wet_grass_prob, 0, 1))

data = pd.DataFrame({'Rain': rain, 'Sprinkler': sprinkler, 'WetGrass': wet_grass})
print("Sample Data:")
display(data.head())

# 2. Define the Bayesian Network structure using the updated DiscreteBayesianNetwork class
# Edges specify the directional dependencies: Rain -> WetGrass and Sprinkler -> WetGrass
model = DiscreteBayesianNetwork([('Rain', 'WetGrass'), ('Sprinkler', 'WetGrass')])

# 3. Fit the model parameters (Conditional Probability Distributions) from data
# Calling fit without an estimator defaults to the standard discrete MLE method
model.fit(data)

# Check if the model is valid
print("\nModel validity check:", model.check_model())

# 4. Perform Inference (Estimate the likelihood of an event)
inference = VariableElimination(model)

# Query: What is the probability of WetGrass given that it is Raining?
query_result = inference.query(variables=['WetGrass'], evidence={'Rain': 1})
print("\nLikelihood of WetGrass given Rain:")
print(query_result)

Sample Data:


,Rain,Sprinkler,WetGrass
0,0,0,0
1,1,0,1
2,1,1,1
3,0,1,1
4,0,1,1



Model validity check: True

Likelihood of WetGrass given Rain:
+-------------+-----------------+
| WetGrass    |   phi(WetGrass) |
+=============+=================+
| WetGrass(0) |          0.0518 |
+-------------+-----------------+
| WetGrass(1) |          0.9482 |
+-------------+-----------------+
